### ENV Setup
---

In [14]:
import os

homebrew_lib = "/opt/homebrew/lib"
homebrew_libffi = "/opt/homebrew/opt/libffi/lib"
os.environ["DYLD_LIBRARY_PATH"] = f"{homebrew_lib}:{homebrew_libffi}:{os.environ.get('DYLD_LIBRARY_PATH', '')}"
os.environ["PKG_CONFIG_PATH"] = f"/opt/homebrew/lib/pkgconfig:{homebrew_libffi}/pkgconfig:{os.environ.get('PKG_CONFIG_PATH', '')}"

from agents import Agent, WebSearchTool, ModelSettings, Runner, trace, function_tool
from openai.types.responses.tool import WebSearchToolFilters
from dotenv import load_dotenv
from IPython.display import display, Markdown
from pydantic import Field, BaseModel
import asyncio
from weasyprint import HTML

In [15]:
load_dotenv(override=True)

True

### Car Parts Search Agent
---

In [16]:
SEARCH_INSTRUCTIONS = "You are a research assistan. Given a search term, you search the \
    web for that term and produce a consice summary of the results. The summary must \
    be 2-3 paragraphs long and less than 300 words. Capture the main points. Write clearly, no \
    need to have complete sentences or good grammar. This will be consumed by someone \
    synthesizing a report, so it's vital you capture the essence and ingore any fluff. \
    Do not include any additional commentary othen than the summary itself."  

In [19]:
search_agent = Agent(
    name="Search Agent",
    instructions=SEARCH_INSTRUCTIONS,
    tools=[
        WebSearchTool(
            user_location={
                "type": "approximate",
                "country": "LV",
                "city": "Riga",
                "region": "Riga",
            }, 
            filters=WebSearchToolFilters(                
                allowed_domains=[
                    "intercars.lv",
                    "trodo.lv",
                    "eparts.lv",
                ]
            ),
            search_context_size="low"
        )
    ],    
    model="gpt-5.5",
    model_settings=ModelSettings(tool_choice="required")
)

In [20]:
search_prompt="budget friendly E92 brake kit"

with trace("Search"):
    result = await Runner.run(search_agent, search_prompt)

display(Markdown(result.final_output))

Search results from allowed Latvian parts sites mainly show **individual brake components**, not a complete “budget E92 brake kit.” Best budget-type matches are on Trodo/Eparts for BMW 3 Coupe E92: ABE front brake pad set around **26.66 €** on Eparts, marked budget class; ABE rear pad set around **27.61 €** on Trodo, also budget class; LPR/FREMAX pad sets around **29–32 €**. ([eparts.lv](https://www.eparts.lv/bremzu-uzliku-komplekts-abe-c1b050abe?utm_source=openai))

For discs, Trodo shows E92-compatible rear brake discs such as **METELLI 23-0814C** around **29.55 € each**, mid-range, solid 10 mm rear disc. ABE brake disc listings also appear for E90/E91/E92/E93, likely the budget-friendly disc option, but the search snippet did not expose full pricing/details. Small supporting parts are cheap: pad accessory kits around **2.63–5.85 €**, brake hoses from about **5.44–8.21 €**, and rear caliper options around **30.68 €**. ([trodo.lv](https://www.trodo.lv/bremzu-diski-metelli-23-0814c?utm_source=openai))

Main takeaway: for a budget E92 brake refresh, build your own kit from **ABE/LPR/FREMAX pads + ABE/METELLI discs + cheap accessory kits**, rather than looking for a pre-bundled kit. Must confirm exact E92 engine/brake system before ordering, because fitment varies by model, axle, disc size, and ATE system details.

### Part Finding Planner Agent
---

In [21]:
SEARCH_AMOUNT = 5

In [22]:
PLANNER_INSTRUCTIONS = f"""
You are an automotive parts search-query planner.

Given a user's request, generate exactly {SEARCH_AMOUNT} highly specific,
short search queries for automotive parts stores and catalogs.

Your queries must be optimized for parts-catalog search, not general web search.

RULES:
- Use precise automotive terminology and catalog-style keywords.
- Include vehicle make, model, chassis/generation (e.g. E92), and year/engine
  when provided and relevant.
- Include the exact part type requested (e.g. brake discs, brake pads, clutch kit).
- Preserve important specifications such as ceramic, drilled, vented, slotted,
  performance, OEM, front, rear, etc.
- Prefer compact keyword combinations over natural-language sentences.
- Do NOT write questions or conversational queries.
- Do NOT use filler such as "best", "please", "I need", "looking for",
  "recommend", "budget friendly", or "what should I buy".
- Do NOT make queries verbose.
- Do NOT invent vehicle specifications or part specifications that are not
  present in the user's request.
- Generate distinct search variants that could find different relevant products.
- When appropriate, vary one important catalog term at a time
  (e.g. "brake discs", "ceramic brake discs", "front brake discs pads").
- Use terminology commonly found in automotive parts catalogs.

Example:

User request:
"budget friendly E92 brake kit"

Good queries:
BMW E92 brake kit
BMW E92 brake discs pads
BMW E92 ceramic brake discs
BMW E92 front brake discs
BMW E92 sport brake kit

Bad queries:
best budget-friendly brake kit for BMW E92
what is the best brake kit for an E92
affordable high-performance ceramic brake system for BMW E92

Output ONLY the {SEARCH_AMOUNT} search queries, one query per line.
"""

In [23]:
class WebSearchItem(BaseModel):
    reason: str = Field(description="Your reasoning for why this search is important to the query.")
    query: str = Field(description="The search term to use for the web search")

class WebSearchPlan(BaseModel):
    searches: list[WebSearchItem] = Field(description="A list of web searches to perform to answer the query.")

In [24]:
planner_agent = Agent(
    name="Planner Agent",
    instructions=PLANNER_INSTRUCTIONS,
    model="gpt-4.1-nano",
    output_type=WebSearchPlan,
)

In [25]:
planner_prompt = "budget friendly E92 brake kit"

with trace("Car parts search"):
    result = await Runner.run(planner_agent, planner_prompt)
    print(result.final_output)

searches=[WebSearchItem(reason='To find cost-effective brake kit options compatible with BMW E92.', query='BMW E92 budget brake kit'), WebSearchItem(reason='To locate affordable brake disc and pad sets for BMW E92.', query='BMW E92 brake discs pads budget'), WebSearchItem(reason='To explore economical ceramic brake disc kits for BMW E92.', query='BMW E92 ceramic brake discs budget'), WebSearchItem(reason='To find rear brake kit options for BMW E92.', query='BMW E92 rear brake kit budget'), WebSearchItem(reason='To locate economical sport brake kits compatible with BMW E92.', query='BMW E92 sport brake kit budget')]


### Report Agent
---

In [28]:
REPORT_INSTRUCTIONS = """
You are a senior automotive research analyst responsible for producing a
professional, decision-oriented report from a user's research query and
research gathered by other agents.

You will receive:
1. The original user query.
2. Research findings from automotive parts stores and other sources.

Your task is to transform the research into a detailed, cohesive HTML report
that can be converted directly into a PDF using WeasyPrint.

REPORT PROCESS:
1. Analyze the original query and identify the user's actual requirements.
2. Organize the available research into a logical report structure.
3. Compare relevant products and clearly distinguish meaningful differences.
4. Identify the best options based on the user's requirements.
5. Highlight important fitment, specifications, prices, availability, and
   other relevant information found in the research.
6. Clearly identify uncertainty, missing information, or specifications that
   could not be verified.
7. Do not invent product specifications, prices, compatibility, brands,
   availability, or other facts that are not present in the research.
8. Prefer concrete facts and comparisons over generic automotive advice.

REPORT STRUCTURE:
Use an appropriate structure for the particular query. When relevant,
include:

- Executive summary
- User requirements
- Vehicle / fitment information
- Search methodology or sources
- Product comparison
- Detailed product analysis
- Price comparison
- Technical specification comparison
- Advantages and disadvantages
- Best options / recommendations
- Important fitment considerations
- Final recommendation
- Sources

Do not force sections that are not relevant to the query.

PRODUCT COMPARISONS:
When comparing automotive parts, prioritize:

- Vehicle make/model/chassis
- Engine and year when relevant
- Front/rear position
- Part type
- Brand and manufacturer
- Part number
- Dimensions
- Material
- Construction
- Performance characteristics
- Certification or standards when available
- Price
- Availability
- Seller
- Product URL
- Fitment confidence

Use tables where they make comparisons easier to understand.

WRITING STYLE:
- Write like a professional automotive research analyst.
- Be factual, precise, and concise.
- Avoid marketing language and unsupported claims.
- Explain technical differences in practical terms.
- Do not repeat the same information unnecessarily.
- Use headings, subheadings, tables, bullet lists, and short paragraphs.
- Make the report useful for someone deciding which product to purchase.
- Target approximately 1,000-2,500 words depending on the amount of available
  research. Do not add filler just to reach a word count.

HTML REQUIREMENTS:
The final output must be a COMPLETE, VALID HTML DOCUMENT.

The HTML must:
- Start with <!DOCTYPE html>.
- Contain <html>, <head>, and <body>.
- Include all required CSS inside a <style> element.
- Be suitable for direct conversion to PDF using WeasyPrint.
- Use A4 page formatting.
- Use print-friendly colors and spacing.
- Use tables for structured product comparisons.
- Use page-break rules where appropriate.
- Avoid JavaScript.
- Do not depend on a hosted webpage or external CSS.
- Do not use Markdown.
- Do not wrap the HTML in Markdown code fences.
- Do not add commentary before or after the HTML.

PDF DESIGN:
Create a clean, professional automotive report.

Use:
- A4 page size
- Approximately 18mm page margins
- Clear typography
- Professional heading hierarchy
- Subtle borders and background colors
- Well-formatted comparison tables
- Highlight boxes for important findings
- Consistent spacing
- Page numbers where practical
- URLs as clickable links
- Avoid excessively large headings or wasted whitespace

If product images are provided in the research, you may include them using
their provided URLs. Do not invent image URLs.

Use CSS suitable for WeasyPrint, for example:

@page {
    size: A4;
    margin: 18mm;
}

Avoid CSS features that require JavaScript or browser-specific rendering.

SOURCE HANDLING:
Every important product claim should be traceable to the provided research.
Where source URLs are available, include them as clickable links in the report.

Do not fabricate citations or URLs.

FINAL OUTPUT:
Return ONLY the complete HTML document.

The HTML should follow this general structure, adapting it to the actual
research:

<!DOCTYPE html>
<html>
<head>
<meta charset="UTF-8">
<title>Automotive Parts Research Report</title>
<style>
@page {
    size: A4;
    margin: 18mm;
}

body {
    font-family: Arial, Helvetica, sans-serif;
    color: #222;
    font-size: 10.5pt;
    line-height: 1.5;
}

h1 {
    font-size: 24pt;
    color: #111;
    margin-bottom: 8px;
}

h2 {
    font-size: 17pt;
    color: #222;
    margin-top: 24px;
    page-break-after: avoid;
}

h3 {
    font-size: 13pt;
    margin-top: 18px;
    page-break-after: avoid;
}

p {
    margin: 6px 0 10px;
}

table {
    width: 100%;
    border-collapse: collapse;
    margin: 12px 0 18px;
    font-size: 9pt;
}

th {
    background: #222;
    color: white;
    font-weight: bold;
    text-align: left;
}

th, td {
    padding: 7px;
    border: 1px solid #ccc;
    vertical-align: top;
}

tr {
    page-break-inside: avoid;
}

.product {
    border: 1px solid #ddd;
    padding: 12px;
    margin: 12px 0;
    page-break-inside: avoid;
}

.price {
    font-size: 16pt;
    font-weight: bold;
    color: #111;
}

.highlight {
    background: #f3f6f8;
    border-left: 4px solid #333;
    padding: 10px 14px;
    margin: 12px 0;
    page-break-inside: avoid;
}

.warning {
    background: #fff4e5;
    border-left: 4px solid #e67e22;
    padding: 10px 14px;
    margin: 12px 0;
    page-break-inside: avoid;
}

a {
    color: #1558a6;
    text-decoration: none;
}

.footer {
    color: #777;
    font-size: 8pt;
    margin-top: 25px;
}
</style>
</head>

<body>

<!-- Generate the actual report here -->

</body>
</html>
"""

In [29]:
class ReportData(BaseModel):
    short_summary: str = Field(description="A short 2-3 sentence summary of the findings.")
    markdown_report: str = Field(description="The final report stored using HTML markup")
    follow_up_questions: str = Field(description="Suggested topics to research further.")

writer_agent = Agent(
    name="Writer Agent",
    instructions=REPORT_INSTRUCTIONS,
    model="gpt-4.1-nano",
    output_type=ReportData,
)

### PDF Generation Agent
---

In [30]:
FILE_GENERATION_INSTRUCTIONS = """
You are a PDF File Generation Agent.

Your task is to convert the provided HTML content into a complete, professional PDF document using the `generate_pdf` tool.

## Input

You will receive an HTML string containing the document's content, structure, and formatting.

## Requirements

1. HTML to PDF conversion
   - Always use the `generate_pdf` tool to generate the PDF.
   - Pass the complete HTML string to the tool without modifying its content.
   - The HTML is the source of truth.
   - Do not summarize, rewrite, remove, or invent any content.

2. Preserve formatting
   - Preserve headings, paragraphs, lists, tables, links, images, alignment, spacing, and other HTML structure.
   - Preserve CSS styling whenever supported by the PDF renderer.
   - Respect print-related CSS such as `@page`, margins, page breaks, and `break-before` / `break-after` rules where supported.
   - Ensure the resulting PDF is readable and suitable for printing or sharing.

3. Document integrity
   - Do not truncate or omit content.
   - Ensure tables and structured content remain readable.
   - Keep the original ordering of all content.
   - Do not add titles, headers, footers, metadata, or other content unless they are already present in the HTML.

4. Tool usage
   - Use the `generate_pdf` tool exactly for PDF generation.
   - Do not attempt to generate the PDF using another method.
   - Do not return the HTML itself.

5. Error handling
   - If PDF generation fails, do not claim that the PDF was successfully created.
   - Return a concise description of the error instead of a file path.

6. Final response
   - If PDF generation succeeds, return ONLY the file path returned by `generate_pdf`.
   - Do not include Markdown, explanations, commentary, status messages, or code blocks.
   - Do not say things such as "PDF generated successfully".
   - The final response must contain only the generated PDF file path.

## Workflow

1. Receive the HTML content.
2. Pass the complete HTML content to `generate_pdf`.
3. Wait for the tool to finish.
4. If successful, return only the returned file path.
5. If unsuccessful, return the error without pretending the file exists.
"""

In [31]:
@function_tool
async def generate_pdf(html: str):
    print("Generating PDF...")
    output_path = "report.pdf"
    HTML(string=html).write_pdf(output_path)
    print("PDF generated")
    return output_path

In [32]:
file_generation_agent = Agent(
    name="File Generation Agent",
    instructions=FILE_GENERATION_INSTRUCTIONS,
    model="gpt-4.1-nano",
    tools=[generate_pdf]
)

### Combining Search Planner and Search Agents
---

In [33]:
async def plan_searches(query: str):
    print("Planning searches...")
    result = await Runner.run(planner_agent, f"Query: {query}")
    print(f"Will perform {len(result.final_output.searches)} searches")
    return result.final_output

async def perform_searches(search_plan: WebSearchPlan):
    print("Searching...")
    tasks = [asyncio.create_task(search(item)) for item in search_plan.searches]
    results = await asyncio.gather(*tasks)
    print("Finished searching")
    return results

async def search(item: WebSearchItem):
    search_prompt = f"Search term: {item.query}, \n Reason for searching: {item.reason}"
    result = await Runner.run(search_agent, search_prompt)
    return result.final_output

In [34]:
async def write_report(query: str, search_results: list[str]):
    print("Generating report...")
    input = f"Original query: {query}, \n Summarized search results: {search_results}"
    result = await Runner.run(writer_agent, input)
    print("Finished writing report")
    return result.final_output

In [35]:
query = "budget friendly E92 brake kit"

with trace("Automted part research"):
    print("Starting reserach...")
    search_plan = await plan_searches(query)
    search_results = await perform_searches(search_plan)
    report = await write_report(query, search_results)
    file_path = await Runner.run(file_generation_agent, report.markdown_report)
    # pdf_file = await generate_pdf(report.markdown_report)
    print(f"Generated report file: {file_path.final_output}")
    print("Research finished!")

Starting reserach...
Planning searches...
Will perform 5 searches
Searching...
Finished searching
Generating report...
Finished writing report
Generating PDF...
PDF generated
Generated report file: report.pdf
Research finished!
